<div style="background-color: #f8fafc; border: 1px solid #d9e2ec; border-left: 4px solid #1f4e79; border-radius: 6px; padding: 14px 18px; margin: 16px 0; color: #1f2933; font-family: Arial, sans-serif; line-height: 1.55;">
  <h3 style="margin: 0 0 8px 0; color: #1f4e79; font-size: 18px;">Notebook 11 —  Feature Store consolidé</h3>
  <div style="color: #4b5563; font-size: 14px;">Ce notebook construit le feature store consolidé de modélisation. Son rôle est de produire un dataset final propre, documenté et directement utilisable par le notebook de modélisation. À ce stade, le pipeline ne cherche plus à inventer de nouvelles variables. Il cherche à stabiliser le contrat d’entrée du modèle.</div>
</div>

<div style="background-color: #f8fafc; border: 1px solid #d9e2ec; border-left: 4px solid #1f4e79; border-radius: 6px; padding: 14px 18px; margin: 16px 0; color: #1f2933; font-family: Arial, sans-serif; line-height: 1.55;">
  <h3 style="margin: 0 0 8px 0; color: #1f4e79; font-size: 18px;">PARAMÈTRES, CHEMINS ET FONCTIONS UTILITAIRES</h3>
  <div style="color: #4b5563; font-size: 14px;">Les entrées sont celles du notebook 10 : base scénarios, métadonnées et dictionnaire de scénarios. Les sorties sont centralisées dans un répertoire dédié au feature store avec taux.</div>
</div>

In [ ]:
# Import
from pathlib import Path
from datetime import datetime
import json
import re
import warnings

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 250)
pd.set_option("display.max_colwidth", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
warnings.filterwarnings("default")

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name.lower() == "notebooks" else CURRENT_DIR
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

RUN_DATE = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
VERSION = "v9_taux"

# Entrées canoniques produites par le notebook 10.
INPUT_BASE_PATH = DATA_PROCESSED / "df_modelisation_scenarios_features.parquet"
INPUT_SCENARIO_METADATA_PATH = DATA_PROCESSED / "df_scenarios_features_metadata.parquet"
INPUT_SCENARIO_FEATURES_PATH = DATA_PROCESSED / "dict_scenarios_features.json"
INPUT_FEATURE_BLOCKS_PATH = OUTPUTS_DIR / "selection_features_scenarios" / "config" / "feature_blocks_selected_v1.json"

# Source complémentaire de taux, utilisée uniquement si les variables de taux sont absentes de la base principale.
TAUX_SOURCE_PATH = DATA_PROCESSED / "df_biens_residentiels_temporel.parquet"

# Sorties du présent notebook.
FS_DIR = OUTPUTS_DIR / "feature_store_modelisation_taux"
REPORTS_DIR = FS_DIR / "reports"
CONFIG_DIR = FS_DIR / "config"
TABLES_DIR = FS_DIR / "tables"

OUTPUT_FEATURE_STORE_PATH = TABLES_DIR / "dataset_modelisation_selected_taux.parquet"
OUTPUT_FEATURE_STORE_SAMPLE_PATH = TABLES_DIR / "dataset_modelisation_selected_taux_sample_100k.csv"
OUTPUT_PREPROCESSING_PLAN_PATH = REPORTS_DIR / "preprocessing_plan_taux.csv"
OUTPUT_QUALITY_REPORT_PATH = REPORTS_DIR / "quality_report_taux.csv"

for p in [FS_DIR, REPORTS_DIR, CONFIG_DIR, TABLES_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Projet        :", PROJECT_ROOT)
print("Date exécution:", RUN_DATE)
print("Entrée base   :", INPUT_BASE_PATH)
print("Entrée scénarios:", INPUT_SCENARIO_FEATURES_PATH)
print("Source taux   :", TAUX_SOURCE_PATH)
print("Dossier sortie:", FS_DIR)


Projet        : C:\Users\club_\OneDrive\13_DOCUMENT\SYSTEME_AIDE_DECISION_IMMOBILIERE
Date exécution: 2026-05-25 11:42:17
Entrée base   : C:\Users\club_\OneDrive\13_DOCUMENT\SYSTEME_AIDE_DECISION_IMMOBILIERE\data\processed\df_modelisation_scenarios_features.parquet
Entrée scénarios: C:\Users\club_\OneDrive\13_DOCUMENT\SYSTEME_AIDE_DECISION_IMMOBILIERE\data\processed\dict_scenarios_features.json
Source taux   : C:\Users\club_\OneDrive\13_DOCUMENT\SYSTEME_AIDE_DECISION_IMMOBILIERE\data\processed\df_biens_residentiels_temporel.parquet
Dossier sortie: C:\Users\club_\OneDrive\13_DOCUMENT\SYSTEME_AIDE_DECISION_IMMOBILIERE\outputs\feature_store_modelisation_taux


In [ ]:
#Reprise fonctions utiltaires

def ensure_dir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path


def first_existing(paths):
    """Retourne le premier chemin existant dans une liste de candidats."""
    existing = [Path(p) for p in paths if Path(p).exists()]
    return existing[0] if existing else None


def read_first_existing(paths, label: str, required: bool = True):
    """Lit le premier parquet existant parmi plusieurs candidats."""
    path = first_existing(paths)
    if path is None:
        msg = f"Aucun fichier trouvé pour {label}. Fichiers testés:\n" + "\n".join(f"- {p}" for p in paths)
        if required:
            raise FileNotFoundError(msg)
        warnings.warn(msg)
        return None, None
    df = pd.read_parquet(path)
    print(f"{label} chargé : {path}")
    print(f"Shape {label} : {df.shape[0]:,} lignes | {df.shape[1]:,} colonnes")
    return df, path


def normalize_keys(df: pd.DataFrame) -> pd.DataFrame:
    """Normalise les clés géographiques pour sécuriser les merges."""
    df = df.copy()
    if "code_iris" in df.columns:
        df["code_iris"] = df["code_iris"].astype("string").str.replace(r"\.0$", "", regex=True).str.zfill(9)
    if "code_commune" in df.columns:
        df["code_commune"] = df["code_commune"].astype("string").str.replace(r"\.0$", "", regex=True).str.zfill(5)
    if "code_departement" in df.columns:
        df["code_departement"] = df["code_departement"].astype("string").str.replace(r"\.0$", "", regex=True).str.zfill(2)
    return df


def safe_merge_m1(base: pd.DataFrame, enrich: pd.DataFrame, key: str, label: str, overwrite: bool = False) -> pd.DataFrame:
    """Merge many-to-one sécurisé : ajoute uniquement les colonnes absentes, sauf overwrite=True."""
    if enrich is None or enrich.empty:
        warnings.warn(f"Table {label} vide : merge ignoré.")
        return base
    if key not in base.columns:
        warnings.warn(f"Clé {key} absente de la base principale : merge {label} ignoré.")
        return base
    if key not in enrich.columns:
        warnings.warn(f"Clé {key} absente de {label} : merge ignoré.")
        return base

    base = normalize_keys(base)
    enrich = normalize_keys(enrich)
    before_cols = set(base.columns)
    enrich = enrich.drop_duplicates(key)

    if overwrite:
        common = [c for c in enrich.columns if c != key and c in base.columns]
        if common:
            base = base.drop(columns=common)

    cols_to_add = [c for c in enrich.columns if c == key or c not in base.columns]
    out = base.merge(enrich[cols_to_add], on=key, how="left", validate="m:1")
    added = [c for c in out.columns if c not in before_cols]

    print(f"Merge {label} sur {key} : +{len(added)} colonnes")
    if added:
        print("Colonnes ajoutées exemple :", added[:30])
    return out


def load_json_first(paths, label: str, required: bool = True):
    path = first_existing(paths)
    if path is None:
        msg = f"Aucun JSON trouvé pour {label}. Fichiers testés:\n" + "\n".join(f"- {p}" for p in paths)
        if required:
            raise FileNotFoundError(msg)
        warnings.warn(msg)
        return None, None
    with open(path, "r", encoding="utf-8") as f:
        payload = json.load(f)
    print(f"{label} chargé : {path}")
    return payload, path


def uniq_features(*blocks):
    merged = []
    for block in blocks:
        merged.extend(block or [])
    return list(dict.fromkeys(merged))

<div style="background-color: #f8fafc; border: 1px solid #d9e2ec; border-left: 4px solid #1f4e79; border-radius: 6px; padding: 14px 18px; margin: 16px 0; color: #1f2933; font-family: Arial, sans-serif; line-height: 1.55;">
  <h3 style="margin: 0 0 8px 0; color: #1f4e79; font-size: 18px;">CHARGEMENT DU CONTRAT NOTEBOOK 10</h3>
  <div style="color: #4b5563; font-size: 14px;">Le contrat est lu depuis les exports du notebook 10 : familles sélectionnées, scénarios et métadonnées. Aucun chemin historique n'est testé afin d'éviter les références à des fichiers non utilisés.</div>
</div>

In [3]:
FEATURE_BLOCKS_SELECTED, SELECTED_JSON = load_json_first(
    [INPUT_FEATURE_BLOCKS_PATH],
    "contrat variables sélectionnées notebook 10",
    required=False,
)

SCENARIO_FEATURES, SCENARIO_JSON = load_json_first(
    [INPUT_SCENARIO_FEATURES_PATH],
    "contrat scénarios notebook 10",
    required=False,
)

SCENARIO_METADATA = {}
SCENARIO_META_JSON = None
if INPUT_SCENARIO_METADATA_PATH.exists():
    scenario_metadata_df = pd.read_parquet(INPUT_SCENARIO_METADATA_PATH)
    if {"scenario", "nb_features"}.issubset(scenario_metadata_df.columns):
        SCENARIO_METADATA = {
            row["scenario"]: {
                k: row[k]
                for k in scenario_metadata_df.columns
                if k != "scenario"
            }
            for _, row in scenario_metadata_df.iterrows()
        }
    print("Métadonnées scénarios chargées :", INPUT_SCENARIO_METADATA_PATH)
else:
    warnings.warn(f"Métadonnées scénarios absentes : {INPUT_SCENARIO_METADATA_PATH}")

if FEATURE_BLOCKS_SELECTED is None:
    warnings.warn("Contrat de familles notebook 10 introuvable : une sélection minimale sera recalculée après chargement de la base.")
if SCENARIO_FEATURES is None:
    warnings.warn("Contrat de scénarios notebook 10 introuvable : des scénarios minimaux seront reconstruits.")


contrat variables sélectionnées notebook 10 chargé : C:\Users\club_\OneDrive\13_DOCUMENT\SYSTEME_AIDE_DECISION_IMMOBILIERE\outputs\selection_features_scenarios\config\feature_blocks_selected_v1.json
contrat scénarios notebook 10 chargé : C:\Users\club_\OneDrive\13_DOCUMENT\SYSTEME_AIDE_DECISION_IMMOBILIERE\data\processed\dict_scenarios_features.json
Métadonnées scénarios chargées : C:\Users\club_\OneDrive\13_DOCUMENT\SYSTEME_AIDE_DECISION_IMMOBILIERE\data\processed\df_scenarios_features_metadata.parquet


<div style="background-color: #f8fafc; border: 1px solid #d9e2ec; border-left: 4px solid #1f4e79; border-radius: 6px; padding: 14px 18px; margin: 16px 0; color: #1f2933; font-family: Arial, sans-serif; line-height: 1.55;">
  <h3 style="margin: 0 0 8px 0; color: #1f4e79; font-size: 18px;">CHARGEMENT DE LA BASE NOTEBOOK 10 ET COMPLÉMENT TAUX</h3>
  <div style="color: #4b5563; font-size: 14px;">La base principale correspond à l'export canonique du notebook 10. Les variables de taux sont contrôlées puis rattachées depuis la source temporelle seulement si elles ne sont pas déjà présentes.</div>
</div>

In [4]:
EXPECTED_MIN_ROWS = 3_500_000
STRICT_ROW_CONTROL = False

if not INPUT_BASE_PATH.exists():
    raise FileNotFoundError(
        f"Base notebook 10 introuvable : {INPUT_BASE_PATH}\n"
        "Exécuter le notebook 10 avant ce notebook."
    )

df = pd.read_parquet(INPUT_BASE_PATH).copy()
df = normalize_keys(df)
MAIN_SOURCE_PATH = INPUT_BASE_PATH
MARKET_SOURCE_PATH = None

if df.shape[0] < EXPECTED_MIN_ROWS:
    msg = f"Volumétrie sous contrôle : {df.shape[0]:,} lignes < {EXPECTED_MIN_ROWS:,}. Vérifier le pointage avant modélisation."
    if STRICT_ROW_CONTROL:
        raise ValueError(msg)
    warnings.warn(msg)

FEATURES_TAUX_CREDIT_REFERENCE = [
    "taux_credit_moyen",
    "variation_taux_1m",
    "variation_taux_3m",
    "taux_credit_roll3",
    "taux_credit_lag1",
    "taux_credit_lag3",
    "score_tension_credit",
]

features_taux_credit = [
    col for col in FEATURES_TAUX_CREDIT_REFERENCE
    if col in df.columns
]

TAUX_SOURCE_USED = None
TAUX_JOIN_KEY = None

if not features_taux_credit:
    if not TAUX_SOURCE_PATH.exists():
        raise FileNotFoundError(
            f"Variables de taux absentes de la base et source taux introuvable : {TAUX_SOURCE_PATH}"
        )

    df_taux_source = pd.read_parquet(TAUX_SOURCE_PATH).copy()
    features_taux_disponibles = [
        col for col in FEATURES_TAUX_CREDIT_REFERENCE
        if col in df_taux_source.columns
    ]

    if not features_taux_disponibles:
        raise ValueError("La source temporelle ne contient aucune variable de taux attendue.")

    if "id_mutation" in df.columns and "id_mutation" in df_taux_source.columns:
        TAUX_JOIN_KEY = "id_mutation"
    elif "annee_mois" in df.columns and "annee_mois" in df_taux_source.columns:
        TAUX_JOIN_KEY = "annee_mois"
        df[TAUX_JOIN_KEY] = df[TAUX_JOIN_KEY].astype("string")
        df_taux_source[TAUX_JOIN_KEY] = df_taux_source[TAUX_JOIN_KEY].astype("string")
    elif "annee_mois_mutation" in df.columns and "annee_mois_mutation" in df_taux_source.columns:
        TAUX_JOIN_KEY = "annee_mois_mutation"
        df[TAUX_JOIN_KEY] = df[TAUX_JOIN_KEY].astype("string")
        df_taux_source[TAUX_JOIN_KEY] = df_taux_source[TAUX_JOIN_KEY].astype("string")
    else:
        raise ValueError("Impossible de rattacher les taux : aucune clé commune id_mutation, annee_mois ou annee_mois_mutation.")

    df_taux_join = (
        df_taux_source[[TAUX_JOIN_KEY] + features_taux_disponibles]
        .drop_duplicates(TAUX_JOIN_KEY)
        .copy()
    )

    df = df.merge(
        df_taux_join,
        on=TAUX_JOIN_KEY,
        how="left",
        validate="many_to_one",
    )

    TAUX_SOURCE_USED = TAUX_SOURCE_PATH
    features_taux_credit = [
        col for col in FEATURES_TAUX_CREDIT_REFERENCE
        if col in df.columns
    ]
    print(f"Features de taux rattachées via {TAUX_JOIN_KEY}.")
else:
    print("Features de taux déjà présentes dans la base notebook 10.")

if not features_taux_credit:
    raise ValueError("Aucune variable de taux disponible après contrôle et rattachement.")

if FEATURE_BLOCKS_SELECTED is not None:
    FEATURE_BLOCKS_SELECTED.setdefault("B6_taux_credit", [])
    FEATURE_BLOCKS_SELECTED["B6_taux_credit"] = list(dict.fromkeys(
        FEATURE_BLOCKS_SELECTED["B6_taux_credit"] + features_taux_credit
    ))

print("Shape après contrôle taux :", f"{df.shape[0]:,} lignes | {df.shape[1]:,} colonnes")
display(pd.DataFrame({"features_taux_credit": features_taux_credit}))
display(
    df[features_taux_credit]
    .isna()
    .mean()
    .sort_values()
    .to_frame("part_valeurs_manquantes")
)



Features de taux déjà présentes dans la base notebook 10.
Shape après contrôle taux : 3,763,971 lignes | 404 colonnes


,features_taux_credit
0,taux_credit_moyen
1,variation_taux_1m
2,variation_taux_3m
3,taux_credit_roll3
4,taux_credit_lag1
5,taux_credit_lag3
6,score_tension_credit


,part_valeurs_manquantes
taux_credit_moyen,0.0000
variation_taux_1m,0.0000
variation_taux_3m,0.0000
taux_credit_roll3,0.0000
taux_credit_lag1,0.0000
taux_credit_lag3,0.0000
score_tension_credit,0.0000


<div style="background-color: #f8fafc; border: 1px solid #d9e2ec; border-left: 4px solid #1f4e79; border-radius: 6px; padding: 14px 18px; margin: 16px 0; color: #1f2933; font-family: Arial, sans-serif; line-height: 1.55;">
  <h3 style="margin: 0 0 8px 0; color: #1f4e79; font-size: 18px;">HARMONISATION DES VARIABLES NÉCESSAIRES À LA MODÉLISATION</h3>
  <div style="color: #4b5563; font-size: 14px;">Cette section reconstruit uniquement des variables simples lorsque leur absence relève d'un renommage ou d'un calcul direct non ambigu. Elle ne crée pas de nouvelles variables métier non validées ; elle sécurise la continuité entre les notebooks producteurs et le fichier final.</div>
</div>

In [5]:
# B1 — caractéristiques du bien.
if "surface_reference" not in df.columns and "surface_reference_model" in df.columns:
    df["surface_reference"] = df["surface_reference_model"]
if "surface_reference_model" not in df.columns and "surface_reference" in df.columns:
    df["surface_reference_model"] = df["surface_reference"]
if "log_surface_reference" not in df.columns and "surface_reference_model" in df.columns:
    df["log_surface_reference"] = np.log1p(pd.to_numeric(df["surface_reference_model"], errors="coerce"))
if "surface_par_piece" not in df.columns and {"surface_reference_model", "nb_pieces_total"}.issubset(df.columns):
    pieces = pd.to_numeric(df["nb_pieces_total"], errors="coerce").replace(0, np.nan)
    df["surface_par_piece"] = pd.to_numeric(df["surface_reference_model"], errors="coerce") / pieces
if "is_maison" not in df.columns and "type_bien" in df.columns:
    df["is_maison"] = df["type_bien"].astype("string").str.lower().str.contains("maison", na=False).astype("int8")
if "is_appartement" not in df.columns and "type_bien" in df.columns:
    df["is_appartement"] = df["type_bien"].astype("string").str.lower().str.contains("appartement", na=False).astype("int8")
if "has_terrain" not in df.columns and "surface_terrain" in df.columns:
    df["has_terrain"] = (pd.to_numeric(df["surface_terrain"], errors="coerce").fillna(0) > 0).astype("int8")
if "ratio_terrain_bati" not in df.columns and {"surface_terrain", "surface_reference_model"}.issubset(df.columns):
    bati = pd.to_numeric(df["surface_reference_model"], errors="coerce").replace(0, np.nan)
    df["ratio_terrain_bati"] = pd.to_numeric(df["surface_terrain"], errors="coerce") / bati
if "intensite_batie" not in df.columns and {"surface_terrain", "surface_reference_model"}.issubset(df.columns):
    terrain = pd.to_numeric(df["surface_terrain"], errors="coerce").replace(0, np.nan)
    df["intensite_batie"] = pd.to_numeric(df["surface_reference_model"], errors="coerce") / terrain

# B2 — temps.
if "date_mutation" in df.columns:
    dt = pd.to_datetime(df["date_mutation"], errors="coerce")
    if "annee_mutation" not in df.columns:
        df["annee_mutation"] = dt.dt.year
    if "annee" not in df.columns:
        df["annee"] = dt.dt.year
    if "mois" not in df.columns:
        df["mois"] = dt.dt.month
    if "mois_mutation" not in df.columns:
        df["mois_mutation"] = dt.dt.month
    if "trimestre" not in df.columns:
        df["trimestre"] = dt.dt.quarter
    if "trimestre_mutation" not in df.columns:
        df["trimestre_mutation"] = dt.dt.quarter
    if "mois_sin" not in df.columns:
        df["mois_sin"] = np.sin(2 * np.pi * dt.dt.month / 12)
    if "mois_cos" not in df.columns:
        df["mois_cos"] = np.cos(2 * np.pi * dt.dt.month / 12)
    if "annee_mois_mutation" not in df.columns:
        df["annee_mois_mutation"] = dt.dt.to_period("M").astype("string")
    if "mois_depuis_debut" not in df.columns:
        min_period = dt.dt.to_period("M").min()
        period = dt.dt.to_period("M")
        df["mois_depuis_debut"] = (period.astype("int64") - min_period.ordinal).where(dt.notna(), np.nan)

# B3 — typologies marché : création de codes si nécessaire.
if "typologie_metier" in df.columns and "typologie_metier_code" not in df.columns:
    df["typologie_metier_code"] = df["typologie_metier"].astype("category").cat.codes.astype("int16")
if "cluster_kmeans" in df.columns and "cluster_kmeans_code" not in df.columns:
    df["cluster_kmeans_code"] = pd.to_numeric(df["cluster_kmeans"], errors="coerce").astype("Int16")

print("Harmonisation terminée.")

Harmonisation terminée.


<div style="background-color: #f8fafc; border: 1px solid #d9e2ec; border-left: 4px solid #1f4e79; border-radius: 6px; padding: 14px 18px; margin: 16px 0; color: #1f2933; font-family: Arial, sans-serif; line-height: 1.55;">
  <h3 style="margin: 0 0 8px 0; color: #1f4e79; font-size: 18px;">APPLICATION DU CONTRAT DE VARIABLES ET CONTRÔLE DES FAMILLES</h3>
  <div style="color: #4b5563; font-size: 14px;">Les familles finales sont recalées sur les colonnes réellement présentes après consolidation. La famille B6 taux est conservée explicitement pour permettre son intégration dans les scénarios de modélisation.</div>
</div>

In [6]:
# Si le contrat notebook 10 est absent, fallback minimal strictement basé sur les colonnes présentes.
if FEATURE_BLOCKS_SELECTED is None:
    b5_candidates = [
        c for c in df.columns
        if str(c).startswith("geo_")
        and re.search(r"_(prix_m2_med|prix_m2_iqr|n|dist_med)$", str(c))
        and not re.search(r"^(ratio_prix_vs_|ecart_abs_prix_vs_|ecart_pct_vs_)", str(c))
    ]
    FEATURE_BLOCKS_SELECTED = {
        "B1_bien": [c for c in ["surface_reference_model", "surface_reference", "log_surface_reference", "classe_surface", "nb_pieces_total", "surface_par_piece", "type_bien", "is_maison", "is_appartement", "surface_terrain", "has_terrain", "ratio_terrain_bati", "intensite_batie"] if c in df.columns],
        "B2_temps": [c for c in ["annee_mutation", "annee", "mois_sin", "mois_cos", "trimestre", "trimestre_mutation", "mois_depuis_debut"] if c in df.columns],
        "B3_territoire": [c for c in ["code_departement", "code_commune", "code_iris", "signal_marche_fiable", "score_liquidite_marche", "score_tension_marche", "score_premium_marche", "score_atypicite_marche", "typologie_metier_code", "cluster_kmeans_code", "dispersion_prix_relative", "transactions_count", "transactions_par_mois"] if c in df.columns],
        "B4_socio_eco": [c for c in ["revenu_median", "revenu_q1", "revenu_q3", "indice_inegalite_revenus", "part_chomage_revenu_disponible", "densite_population_km2", "part_population_15_29", "part_population_65_plus", "part_logements_vacants", "part_residences_secondaires"] if c in df.columns],
        "B5_comparables": b5_candidates[:40],
        "B6_taux_credit": features_taux_credit,
    }

FEATURE_BLOCKS_SELECTED.setdefault("B6_taux_credit", features_taux_credit)
FEATURE_BLOCKS_SELECTED["B6_taux_credit"] = [c for c in FEATURE_BLOCKS_SELECTED["B6_taux_credit"] if c in df.columns]

selected_blocks_final = {
    fam: [c for c in cols if c in df.columns]
    for fam, cols in FEATURE_BLOCKS_SELECTED.items()
}
missing_blocks_final = {
    fam: [c for c in cols if c not in df.columns]
    for fam, cols in FEATURE_BLOCKS_SELECTED.items()
}
selected_features = uniq_features(*selected_blocks_final.values())

block_audit = pd.DataFrame([
    {
        "famille": fam,
        "nb_retenues_contrat": len(FEATURE_BLOCKS_SELECTED.get(fam, [])),
        "nb_presentes_apres_consolidation": len(selected_blocks_final.get(fam, [])),
        "variables_presentes": ", ".join(selected_blocks_final.get(fam, [])),
        "variables_absentes": ", ".join(missing_blocks_final.get(fam, [])),
    }
    for fam in FEATURE_BLOCKS_SELECTED.keys()
])

display(block_audit)
block_audit.to_csv(REPORTS_DIR / "audit_familles_selectionnees_taux.csv", index=False)

print("Nombre total de features sélectionnées présentes :", len(selected_features))
print("Features taux retenues :", selected_blocks_final.get("B6_taux_credit", []))


,famille,nb_retenues_contrat,nb_presentes_apres_consolidation,variables_presentes,variables_absentes
0,B1_bien,17,17,"classe_surface, has_dependance, is_appartement, is_maison, log_surface_reference, nb_dependances, pieces_info_disponible, segment_residentiel_surface, surface_reference, terrain_info_disponible, t...",
1,B2_temps,15,15,"annee, annee_mutation, mois_cos, mois_depuis_debut, mois_sin, trimestre, trimestre_mutation, score_momentum_lag1, score_risque_temporel_lag1, score_tension_temporelle_lag1, volume_tx_lag1, momentu...",
2,B3_territoire,8,8,"code_commune, code_departement, code_iris, score_atypicite_marche, score_liquidite_marche, signal_marche_fiable, transactions_count, transactions_par_mois",
3,B4_socio_eco,12,12,"indice_inegalite_revenus, part_chomage_revenu_disponible, revenu_median, revenu_q1, revenu_q3, densite_population_km2, taille_menage_moyenne, part_population_15_29, part_population_65_plus, densit...",
4,B5_comparables,30,30,"commune_nb_ventes_passe, commune_type_nb_ventes_passe, iris_nb_ventes_passe, commune_prix_m2_med_passe, commune_type_prix_m2_med_passe, iris_prix_m2_med_passe, geo_1000m_12m_all_nb, geo_1000m_24m_...",
5,B6_taux_credit,7,7,"score_tension_credit, taux_credit_lag1, taux_credit_lag3, taux_credit_moyen, taux_credit_roll3, variation_taux_1m, variation_taux_3m",


Nombre total de features sélectionnées présentes : 89
Features taux retenues : ['score_tension_credit', 'taux_credit_lag1', 'taux_credit_lag3', 'taux_credit_moyen', 'taux_credit_roll3', 'variation_taux_1m', 'variation_taux_3m']


<div style="background-color: #f8fafc; border: 1px solid #d9e2ec; border-left: 4px solid #1f4e79; border-radius: 6px; padding: 14px 18px; margin: 16px 0; color: #1f2933; font-family: Arial, sans-serif; line-height: 1.55;">
  <h3 style="margin: 0 0 8px 0; color: #1f4e79; font-size: 18px;">CONSTRUCTION DES SCÉNARIOS COMPATIBLES MODÉLISATION</h3>
  <div style="color: #4b5563; font-size: 14px;">Les scénarios du notebook 10 sont recalés sur le dataset final. Si nécessaire, les déclinaisons avec taux sont ajoutées sans modifier les scénarios existants.</div>
</div>

In [7]:
B1 = selected_blocks_final.get("B1_bien", [])
B2 = selected_blocks_final.get("B2_temps", [])
B3 = selected_blocks_final.get("B3_territoire", [])
B4 = selected_blocks_final.get("B4_socio_eco", [])
B5 = selected_blocks_final.get("B5_comparables", [])
B6 = selected_blocks_final.get("B6_taux_credit", features_taux_credit)

if SCENARIO_FEATURES is None:
    SCENARIO_FEATURES = {
        "S_B1_only": uniq_features(B1),
        "S_B1_B2": uniq_features(B1, B2),
        "S_B1_B3": uniq_features(B1, B3),
        "S_B1_B4": uniq_features(B1, B4),
        "S_B1_B5": uniq_features(B1, B5),
        "S_B1_B2_B3": uniq_features(B1, B2, B3),
        "S_B1_B2_B5": uniq_features(B1, B2, B5),
        "S_B1_B2_B3_B4": uniq_features(B1, B2, B3, B4),
        "S_FULL_SELECTED": uniq_features(B1, B2, B3, B4, B5),
        "S_B2_only": uniq_features(B2),
        "S_B3_only": uniq_features(B3),
        "S_B4_only": uniq_features(B4),
        "S_B5_only": uniq_features(B5),
    }

allowed = set(selected_features)
scenario_features_final = {
    scen: [c for c in cols if c in allowed and c in df.columns]
    for scen, cols in SCENARIO_FEATURES.items()
}

# Sécurisation : scénarios structurants hors taux.
scenario_features_final.setdefault("S_B1_only", uniq_features(B1))
scenario_features_final.setdefault("S_B1_B2", uniq_features(B1, B2))
scenario_features_final.setdefault("S_B1_B3", uniq_features(B1, B3))
scenario_features_final.setdefault("S_B1_B4", uniq_features(B1, B4))
scenario_features_final.setdefault("S_B1_B5", uniq_features(B1, B5))
scenario_features_final.setdefault("S_B1_B2_B3", uniq_features(B1, B2, B3))
scenario_features_final.setdefault("S_B1_B2_B5", uniq_features(B1, B2, B5))
scenario_features_final.setdefault("S_B1_B2_B3_B4", uniq_features(B1, B2, B3, B4))
scenario_features_final.setdefault("S_FULL_SELECTED", uniq_features(B1, B2, B3, B4, B5))

# Complément taux : famille pure et déclinaisons _avec_taux.
scenario_features_final.setdefault("S_B6_taux_only", uniq_features(B6))
for scenario_name, feature_list in list(scenario_features_final.items()):
    if scenario_name.endswith("_avec_taux") or scenario_name == "S_B6_taux_only":
        continue
    scenario_features_final.setdefault(f"{scenario_name}_avec_taux", uniq_features(feature_list, B6))

scenario_summary = pd.DataFrame([
    {
        "scenario": scen,
        "objectif": SCENARIO_METADATA.get(scen, ""),
        "nb_features": len(cols),
        "integre_taux": any(feature in features_taux_credit for feature in cols),
        "features": ", ".join(cols),
    }
    for scen, cols in scenario_features_final.items()
])

display(scenario_summary)
scenario_summary.to_csv(REPORTS_DIR / "scenario_features_final_taux.csv", index=False)


,scenario,objectif,nb_features,integre_taux,features
0,S_B1_only,"{'nb_features': 17, 'integre_taux': False, 'target_col': 'log_prix_m2'}",17,False,"classe_surface, has_dependance, is_appartement, is_maison, log_surface_reference, nb_dependances, pieces_info_disponible, segment_residentiel_surface, surface_reference, terrain_info_disponible, t..."
1,S_B1_B2,"{'nb_features': 32, 'integre_taux': False, 'target_col': 'log_prix_m2'}",32,False,"classe_surface, has_dependance, is_appartement, is_maison, log_surface_reference, nb_dependances, pieces_info_disponible, segment_residentiel_surface, surface_reference, terrain_info_disponible, t..."
2,S_B1_B3,"{'nb_features': 25, 'integre_taux': False, 'target_col': 'log_prix_m2'}",25,False,"classe_surface, has_dependance, is_appartement, is_maison, log_surface_reference, nb_dependances, pieces_info_disponible, segment_residentiel_surface, surface_reference, terrain_info_disponible, t..."
3,S_B1_B4,"{'nb_features': 29, 'integre_taux': False, 'target_col': 'log_prix_m2'}",29,False,"classe_surface, has_dependance, is_appartement, is_maison, log_surface_reference, nb_dependances, pieces_info_disponible, segment_residentiel_surface, surface_reference, terrain_info_disponible, t..."
4,S_B1_B5,"{'nb_features': 47, 'integre_taux': False, 'target_col': 'log_prix_m2'}",47,False,"classe_surface, has_dependance, is_appartement, is_maison, log_surface_reference, nb_dependances, pieces_info_disponible, segment_residentiel_surface, surface_reference, terrain_info_disponible, t..."
5,S_B1_B2_B3,"{'nb_features': 40, 'integre_taux': False, 'target_col': 'log_prix_m2'}",40,False,"classe_surface, has_dependance, is_appartement, is_maison, log_surface_reference, nb_dependances, pieces_info_disponible, segment_residentiel_surface, surface_reference, terrain_info_disponible, t..."
6,S_B1_B2_B5,"{'nb_features': 62, 'integre_taux': False, 'target_col': 'log_prix_m2'}",62,False,"classe_surface, has_dependance, is_appartement, is_maison, log_surface_reference, nb_dependances, pieces_info_disponible, segment_residentiel_surface, surface_reference, terrain_info_disponible, t..."
7,S_B1_B2_B3_B4,"{'nb_features': 52, 'integre_taux': False, 'target_col': 'log_prix_m2'}",52,False,"classe_surface, has_dependance, is_appartement, is_maison, log_surface_reference, nb_dependances, pieces_info_disponible, segment_residentiel_surface, surface_reference, terrain_info_disponible, t..."
8,S_FULL_SELECTED,"{'nb_features': 82, 'integre_taux': False, 'target_col': 'log_prix_m2'}",82,False,"classe_surface, has_dependance, is_appartement, is_maison, log_surface_reference, nb_dependances, pieces_info_disponible, segment_residentiel_surface, surface_reference, terrain_info_disponible, t..."
9,S_B2_only,"{'nb_features': 15, 'integre_taux': False, 'target_col': 'log_prix_m2'}",15,False,"annee, annee_mutation, mois_cos, mois_depuis_debut, mois_sin, trimestre, trimestre_mutation, score_momentum_lag1, score_risque_temporel_lag1, score_tension_temporelle_lag1, volume_tx_lag1, momentu..."


<div style="background-color: #f8fafc; border: 1px solid #d9e2ec; border-left: 4px solid #1f4e79; border-radius: 6px; padding: 14px 18px; margin: 16px 0; color: #1f2933; font-family: Arial, sans-serif; line-height: 1.55;">
  <h3 style="margin: 0 0 8px 0; color: #1f4e79; font-size: 18px;">DATASET FINAL MODEL-READY</h3>
  <div style="color: #4b5563; font-size: 14px;">Le fichier final conserve la cible, les clés utiles à l'analyse des erreurs et les variables sélectionnées. Les valeurs manquantes sont conservées : leur traitement appartient au pipeline de modélisation afin d'éviter de figer prématurément une stratégie d'imputation.</div>
</div>

In [8]:
TARGET_CANDIDATES = ["log_prix_m2", "prix_m2_cible", "prix_m2"]
TARGET = next((c for c in TARGET_CANDIDATES if c in df.columns), None)
if TARGET is None:
    raise ValueError("Aucune cible détectée parmi log_prix_m2 / prix_m2_cible / prix_m2.")

ID_COLUMNS = [
    c for c in [
        "id_mutation", "date_mutation", "code_departement", "code_commune", "code_iris",
        "nom_commune", "nom_iris", "latitude", "longitude", "type_bien",
        "typologie_metier", "cluster_kmeans",
    ]
    if c in df.columns
]

FINAL_COLUMNS = uniq_features(ID_COLUMNS, [TARGET], selected_features)
model_df = df[FINAL_COLUMNS].copy()
model_df = model_df.loc[:, ~model_df.columns.duplicated()]
model_df = model_df.replace([np.inf, -np.inf], np.nan)

# Harmonisation cible : le notebook de modélisation attend en priorité log_prix_m2.
if TARGET != "log_prix_m2":
    if TARGET == "prix_m2" and "log_prix_m2" not in model_df.columns:
        model_df["log_prix_m2"] = np.log(pd.to_numeric(model_df["prix_m2"], errors="coerce"))
        TARGET = "log_prix_m2"
    elif TARGET == "prix_m2_cible" and "log_prix_m2" not in model_df.columns:
        model_df["log_prix_m2"] = np.log(pd.to_numeric(model_df["prix_m2_cible"], errors="coerce"))
        TARGET = "log_prix_m2"

numeric_features = [c for c in selected_features if c in model_df.columns and pd.api.types.is_numeric_dtype(model_df[c])]
categorical_features = [c for c in selected_features if c in model_df.columns and not pd.api.types.is_numeric_dtype(model_df[c])]

quality_report = pd.DataFrame([
    {"controle": "source_principale", "valeur": str(MAIN_SOURCE_PATH)},
    {"controle": "source_taux", "valeur": None if TAUX_SOURCE_USED is None else str(TAUX_SOURCE_USED)},
    {"controle": "cle_rattachement_taux", "valeur": TAUX_JOIN_KEY},
    {"controle": "lignes_finales", "valeur": f"{model_df.shape[0]:,}"},
    {"controle": "colonnes_finales", "valeur": f"{model_df.shape[1]:,}"},
    {"controle": "target", "valeur": TARGET},
    {"controle": "features_selectionnees", "valeur": len(selected_features)},
    {"controle": "features_taux", "valeur": len([c for c in features_taux_credit if c in model_df.columns])},
    {"controle": "features_numeriques", "valeur": len(numeric_features)},
    {"controle": "features_categorielles", "valeur": len(categorical_features)},
] + [
    {"controle": f"features_{fam}", "valeur": len(cols)}
    for fam, cols in selected_blocks_final.items()
])

display(quality_report)
quality_report.to_csv(OUTPUT_QUALITY_REPORT_PATH, index=False)

print("Dataset final :", f"{model_df.shape[0]:,} lignes | {model_df.shape[1]:,} colonnes")


,controle,valeur
0,source_principale,C:\Users\club_\OneDrive\13_DOCUMENT\SYSTEME_AIDE_DECISION_IMMOBILIERE\data\processed\df_modelisation_scenarios_features.parquet
1,source_taux,None
2,cle_rattachement_taux,None
3,lignes_finales,"3,763,971"
4,colonnes_finales,95
5,target,log_prix_m2
6,features_selectionnees,89
7,features_taux,7
8,features_numeriques,82
9,features_categorielles,7


Dataset final : 3,763,971 lignes | 95 colonnes


<div style="background-color: #f8fafc; border: 1px solid #d9e2ec; border-left: 4px solid #1f4e79; border-radius: 6px; padding: 14px 18px; margin: 16px 0; color: #1f2933; font-family: Arial, sans-serif; line-height: 1.55;">
  <h3 style="margin: 0 0 8px 0; color: #1f4e79; font-size: 18px;">Chargement du contrat de features</h3>
  <div style="color: #4b5563; font-size: 14px;">Cette section charge le contrat de features produit par le notebook 10. Ce contrat contient : les blocs B1 à B6 ; les variables retenues par famille ; les scénarios de modélisation ; les métadonnées associées. C’est ce contrat qui gouverne la construction du feature store final.</div>
</div>

<div style="background-color: #f8fafc; border: 1px solid #d9e2ec; border-left: 4px solid #1f4e79; border-radius: 6px; padding: 14px 18px; margin: 16px 0; color: #1f2933; font-family: Arial, sans-serif; line-height: 1.55;">
  <h3 style="margin: 0 0 8px 0; color: #1f4e79; font-size: 18px;">PLAN DE PRÉ-PROCESSING POUR MODÉLISATION</h3>
  <div style="color: #4b5563; font-size: 14px;">Cette étape produit un plan de pré-processing exploitable par le notebook de modélisation : imputation médiane pour les variables numériques, imputation du mode puis encodage pour les variables catégorielles. Le notebook ne transforme pas encore les variables afin de conserver une séparation nette entre préparation et entraînement.</div>
</div>

In [9]:
preprocessing_report = []
for fam, cols in selected_blocks_final.items():
    for col in cols:
        if col not in model_df.columns:
            continue
        s = model_df[col]
        preprocessing_report.append({
            "famille": fam,
            "variable": col,
            "dtype": str(s.dtype),
            "is_numeric": pd.api.types.is_numeric_dtype(s),
            "pct_manquant": float(s.isna().mean()),
            "nunique": int(s.nunique(dropna=True)),
            "suggested_preprocessing": (
                "median_imputer" if pd.api.types.is_numeric_dtype(s)
                else "most_frequent_imputer_plus_onehot_or_catboost_native"
            ),
        })

preprocessing_report = pd.DataFrame(preprocessing_report)
if not preprocessing_report.empty:
    preprocessing_report = preprocessing_report.sort_values(["famille", "is_numeric", "pct_manquant", "variable"], ascending=[True, False, True, True])

display(preprocessing_report)
preprocessing_report.to_csv(OUTPUT_PREPROCESSING_PLAN_PATH, index=False)


,famille,variable,dtype,is_numeric,pct_manquant,nunique,suggested_preprocessing
1,B1_bien,has_dependance,float64,True,0.0000,2,median_imputer
2,B1_bien,is_appartement,int32,True,0.0000,2,median_imputer
3,B1_bien,is_maison,int32,True,0.0000,2,median_imputer
4,B1_bien,log_surface_reference,float64,True,0.0000,33488,median_imputer
5,B1_bien,nb_dependances,int64,True,0.0000,60,median_imputer
...,...,...,...,...,...,...,...
84,B6_taux_credit,taux_credit_lag3,Float64,True,0.0000,52,median_imputer
85,B6_taux_credit,taux_credit_moyen,Float64,True,0.0000,50,median_imputer
86,B6_taux_credit,taux_credit_roll3,float64,True,0.0000,56,median_imputer
87,B6_taux_credit,variation_taux_1m,Float64,True,0.0000,40,median_imputer


<div style="background-color: #f8fafc; border: 1px solid #d9e2ec; border-left: 4px solid #1f4e79; border-radius: 6px; padding: 14px 18px; margin: 16px 0; color: #1f2933; font-family: Arial, sans-serif; line-height: 1.55;">
  <h3 style="margin: 0 0 8px 0; color: #1f4e79; font-size: 18px;">Audit des variables retenues pour la modélisation</h3>
  <div style="color: #4b5563; font-size: 14px;">Le référentiel final contient 89 variables réparties sur les six familles de features du pipeline : caractéristiques du bien ; temporalité ; contexte territorial ; variables socio-économiques ; comparables immobiliers ; taux de crédit. La qualité globale des données apparaît très satisfaisante : absence quasi totale de valeurs manquantes ; forte diversité des variables numériques ; bonne couverture des signaux métier et macroéconomiques. Les variables binaires et structurelles (`is_maison`, `is_appartement`, `has_dependance`) sont correctement typées et directement exploitables. Les variables continues présentent une granularité élevée : surfaces ; comparables géographiques ; scores temporels ; taux de crédit. Le prétraitement proposé reste cohérent et homogène : imputation médiane pour les variables numériques ; préparation compatible avec les pipelines sklearn classiques. L’ensemble constitue un feature store robuste, interprétable et adapté à des scénarios de modélisation immobilière avancée.</div>
</div>

<div style="background-color: #f8fafc; border: 1px solid #d9e2ec; border-left: 4px solid #1f4e79; border-radius: 6px; padding: 14px 18px; margin: 16px 0; color: #1f2933; font-family: Arial, sans-serif; line-height: 1.55;">
  <h3 style="margin: 0 0 8px 0; color: #1f4e79; font-size: 18px;">EXPORTS POUR MODÉLISATION</h3>
  <div style="color: #4b5563; font-size: 14px;">Les exports sont écrits dans le répertoire dédié au feature store avec taux. Ils incluent le dataset model-ready, les familles sélectionnées, les scénarios recalés, le plan de pré-processing et le manifeste d'exécution.</div>
</div>

In [10]:
model_df.to_parquet(OUTPUT_FEATURE_STORE_PATH, index=False)
model_df.head(100_000).to_csv(OUTPUT_FEATURE_STORE_SAMPLE_PATH, index=False)

with open(CONFIG_DIR / "feature_blocks_selected_taux.json", "w", encoding="utf-8") as f:
    json.dump(selected_blocks_final, f, ensure_ascii=False, indent=2)
with open(CONFIG_DIR / "scenario_features_taux.json", "w", encoding="utf-8") as f:
    json.dump(scenario_features_final, f, ensure_ascii=False, indent=2)
with open(CONFIG_DIR / "numeric_features_taux.json", "w", encoding="utf-8") as f:
    json.dump(numeric_features, f, ensure_ascii=False, indent=2)
with open(CONFIG_DIR / "categorical_features_taux.json", "w", encoding="utf-8") as f:
    json.dump(categorical_features, f, ensure_ascii=False, indent=2)

manifest = {
    "version": VERSION,
    "run_date": RUN_DATE,
    "main_source_path": str(MAIN_SOURCE_PATH),
    "scenario_features_path": None if SCENARIO_JSON is None else str(SCENARIO_JSON),
    "selection_contract_path": None if SELECTED_JSON is None else str(SELECTED_JSON),
    "taux_source_path": None if TAUX_SOURCE_USED is None else str(TAUX_SOURCE_USED),
    "taux_join_key": TAUX_JOIN_KEY,
    "features_taux_credit": features_taux_credit,
    "target": TARGET,
    "rows_final": int(model_df.shape[0]),
    "columns_final": int(model_df.shape[1]),
    "feature_blocks_selected": selected_blocks_final,
    "scenario_features": scenario_features_final,
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "exports": {
        "dataset_modelisation_selected_taux": str(OUTPUT_FEATURE_STORE_PATH),
        "sample_csv_100k": str(OUTPUT_FEATURE_STORE_SAMPLE_PATH),
        "feature_blocks_selected_taux": str(CONFIG_DIR / "feature_blocks_selected_taux.json"),
        "scenario_features_taux": str(CONFIG_DIR / "scenario_features_taux.json"),
        "preprocessing_plan_taux": str(OUTPUT_PREPROCESSING_PLAN_PATH),
        "quality_report_taux": str(OUTPUT_QUALITY_REPORT_PATH),
    },
}

with open(CONFIG_DIR / "feature_store_manifest_taux.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

print("Export dataset      :", OUTPUT_FEATURE_STORE_PATH)
print("Export scénarios    :", CONFIG_DIR / "scenario_features_taux.json")
print("Export manifeste    :", CONFIG_DIR / "feature_store_manifest_taux.json")


Export dataset      : C:\Users\club_\OneDrive\13_DOCUMENT\SYSTEME_AIDE_DECISION_IMMOBILIERE\outputs\feature_store_modelisation_taux\tables\dataset_modelisation_selected_taux.parquet
Export scénarios    : C:\Users\club_\OneDrive\13_DOCUMENT\SYSTEME_AIDE_DECISION_IMMOBILIERE\outputs\feature_store_modelisation_taux\config\scenario_features_taux.json
Export manifeste    : C:\Users\club_\OneDrive\13_DOCUMENT\SYSTEME_AIDE_DECISION_IMMOBILIERE\outputs\feature_store_modelisation_taux\config\feature_store_manifest_taux.json


<div style="background-color: #f8fafc; border: 1px solid #d9e2ec; border-left: 4px solid #1f4e79; border-radius: 6px; padding: 14px 18px; margin: 16px 0; color: #1f2933; font-family: Arial, sans-serif; line-height: 1.55;">
  <h3 style="margin: 0 0 8px 0; color: #1f4e79; font-size: 18px;">CONCLUSION OPÉRATIONNELLE</h3>
  <div style="color: #4b5563; font-size: 14px;">Le feature store consolidé est prêt pour la modélisation. Il consomme les exports du notebook 10, contrôle la présence des variables de taux, les rattache si nécessaire, conserve la logique de sélection par famille et produit les fichiers model-ready. dataset_modelisation_selected_taux.parquet  : actif principal avec taux. scenario_features_taux.json  : scénarios recalés, incluant les déclinaisons avec taux. feature_store_manifest_taux.json  : traçabilité des sources, contrôles et exports.</div>
</div>

<div style="background-color: #f8fafc; border: 1px solid #d9e2ec; border-left: 4px solid #1f4e79; border-radius: 6px; padding: 14px 18px; margin: 16px 0; color: #1f2933; font-family: Arial, sans-serif; line-height: 1.55;">
  <h3 style="margin: 0 0 8px 0; color: #1f4e79; font-size: 18px;">Synthèse du notebook 11</h3>
  <div style="color: #4b5563; font-size: 14px;">Ce notebook construit le feature store final de modélisation. Il consomme : la base enrichie issue des notebooks précédents ; les blocs de features sélectionnés ; les scénarios définis au notebook 10. Il produit : un dataset final de modélisation ; un dictionnaire de scénarios ; des métadonnées ; un rapport de pré-processing ; un manifeste d’exécution. Sa contribution principale est de figer le contrat d’entrée du notebook 12. À ce stade, le pipeline a terminé la construction et la gouvernance des features. La suite consiste à tester empiriquement les scénarios, comparer les modèles et mesurer l’apport réel des blocs B1 à B6.</div>
</div>